[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch01/NN_DL_ch01_Activations/NN_DL_ch01_Activations.ipynb)

# NN_DL_ch01_Activations

**Activation Functions, Derivatives & Universal Approximation Theorem**

Visualize all major activation functions with their derivatives, and demonstrate the UAT by approximating sin(x) with increasing network width.

In [ ]:
!pip install -q torch matplotlib scipy

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Sigmoid & Tanh with Derivatives
x = np.linspace(-6, 6, 500)

sigmoid = 1 / (1 + np.exp(-x))
sigmoid_d = sigmoid * (1 - sigmoid)
tanh_v = np.tanh(x)
tanh_d = 1 - tanh_v**2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x, sigmoid,   color=MAIN_BLUE, lw=2.5, label='sigmoid(x)')
axes[0].plot(x, sigmoid_d, color=IDA_RED,   lw=2.5, label="sigmoid'(x)", ls='--')
axes[0].axhline(0, color='grey', lw=0.5); axes[0].axvline(0, color='grey', lw=0.5)
axes[0].set_title('Sigmoid', fontweight='bold'); axes[0].legend(); axes[0].set_ylim(-0.1, 1.1)

axes[1].plot(x, tanh_v, color=MAIN_BLUE, lw=2.5, label='tanh(x)')
axes[1].plot(x, tanh_d, color=IDA_RED,   lw=2.5, label="tanh'(x)", ls='--')
axes[1].axhline(0, color='grey', lw=0.5); axes[1].axvline(0, color='grey', lw=0.5)
axes[1].set_title('Tanh', fontweight='bold'); axes[1].legend(); axes[1].set_ylim(-1.2, 1.5)

plt.suptitle('Sigmoid & Tanh Activation Functions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch01_sigmoid')

In [ ]:
# All Activations Overview
x = np.linspace(-4, 4, 500)
from scipy.special import erf

activations = {
    'Sigmoid':  1 / (1 + np.exp(-x)),
    'Tanh':     np.tanh(x),
    'ReLU':     np.maximum(0, x),
    'LeakyReLU': np.where(x > 0, x, 0.1 * x),
    'ELU':      np.where(x > 0, x, np.exp(x) - 1),
    'GELU':     x * 0.5 * (1 + erf(x / np.sqrt(2))),
    'Swish':    x / (1 + np.exp(-x)),
    'Softplus': np.log1p(np.exp(x)),
}

colors = [MAIN_BLUE, IDA_RED, FOREST, CRIMSON, AMBER, '#9C27B0', '#00BCD4', '#FF9800']
fig, ax = plt.subplots(figsize=(12, 7))
for (name, vals), color in zip(activations.items(), colors):
    ax.plot(x, vals, lw=2.5, label=name, color=color)
ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Comparison of Activation Functions', fontweight='bold', fontsize=15)
ax.legend(ncol=2, fontsize=11); ax.set_ylim(-2, 5)
plt.tight_layout()
save_fig('nn_ch01_activations')

In [ ]:
# ReLU Variants with Derivatives
x = np.linspace(-4, 4, 500)

relu = np.maximum(0, x); relu_d = (x > 0).astype(float)
leaky = np.where(x > 0, x, 0.1 * x); leaky_d = np.where(x > 0, 1.0, 0.1)
elu = np.where(x > 0, x, np.exp(x) - 1); elu_d = np.where(x > 0, 1.0, np.exp(x))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, f, fd), color in zip(axes,
    [('ReLU', relu, relu_d), ('LeakyReLU', leaky, leaky_d), ('ELU', elu, elu_d)],
    [MAIN_BLUE, FOREST, CRIMSON]):
    ax.plot(x, f, color=color, lw=2.5, label=f'{name}(x)')
    ax.plot(x, fd, color=color, lw=2, ls='--', alpha=0.7, label=f"{name}'(x)")
    ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
    ax.set_title(name, fontweight='bold'); ax.legend(); ax.set_ylim(-1.5, 4)
plt.suptitle('ReLU Variants and Their Derivatives', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch01_relu_variants')

In [ ]:
# UAT: Bump Functions
x = np.linspace(-5, 5, 500)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

s1 = 1 / (1 + np.exp(-3 * (x - 1)))
axes[0].plot(x, s1, color=MAIN_BLUE, lw=2.5)
axes[0].set_title('Single Sigmoid Unit', fontweight='bold')
axes[0].axhline(0, color='grey', lw=0.5)

s2a = 1 / (1 + np.exp(-5 * x))
s2b = 1 / (1 + np.exp(-5 * (x - 2)))
bump = s2a - s2b
axes[1].plot(x, s2a, color=MAIN_BLUE, lw=1.5, ls='--', alpha=0.5, label='$\\sigma(5x)$')
axes[1].plot(x, s2b, color=IDA_RED,   lw=1.5, ls='--', alpha=0.5, label='$\\sigma(5(x-2))$')
axes[1].plot(x, bump, color=FOREST,   lw=2.5, label='Bump = difference')
axes[1].set_title('Two Sigmoids -> Bump', fontweight='bold'); axes[1].legend()

combined = np.zeros_like(x)
for c, h in zip([-3, -1, 1, 3], [0.5, 1.2, -0.8, 0.7]):
    sa = 1 / (1 + np.exp(-8 * (x - (c - 0.5))))
    sb = 1 / (1 + np.exp(-8 * (x - (c + 0.5))))
    combined += h * (sa - sb)
axes[2].plot(x, combined, color=MAIN_BLUE, lw=2.5)
axes[2].fill_between(x, combined, alpha=0.15, color=MAIN_BLUE)
axes[2].set_title('Sum of Bumps -> Arbitrary Shape', fontweight='bold')

plt.suptitle('UAT: Building Functions from Sigmoid Bumps', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch01_uat_bumps')

In [ ]:
# UAT: Approximate sin(x) with Increasing Width
import torch.nn as nn
import torch.optim as optim

x_t = torch.linspace(-2 * np.pi, 2 * np.pi, 500).unsqueeze(1)
y_t = torch.sin(x_t)

class UAT_Net(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, width), nn.ReLU(), nn.Linear(width, 1))
    def forward(self, x):
        return self.net(x)

widths = [2, 5, 20, 100]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, w in zip(axes, widths):
    torch.manual_seed(42)
    net = UAT_Net(w)
    opt = optim.Adam(net.parameters(), lr=0.01)
    for _ in range(2000):
        pred = net(x_t)
        loss = nn.MSELoss()(pred, y_t)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        y_pred = net(x_t).numpy()
    ax.plot(x_t.numpy(), y_t.numpy(), color='grey', lw=2, label='sin(x)', alpha=0.7)
    ax.plot(x_t.numpy(), y_pred, color=MAIN_BLUE, lw=2.5, label=f'Width={w}')
    ax.set_title(f'Width = {w} (MSE={loss.item():.4f})', fontweight='bold')
    ax.legend(fontsize=10); ax.set_ylim(-1.5, 1.5)

plt.suptitle('Universal Approximation: sin(x) with Increasing Width', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch01_uat_approximation')

## Summary

Visualized all major activation functions with derivatives and demonstrated the **Universal Approximation Theorem** by showing wider networks approximate sin(x) better.